# Anima LoRA CLI · Colab T4

GPU 런타임 선택 → 설치 → 데이터 → TOML → 모델·전처리 → smoke test → TensorBoard → 본 학습.

- Python 3.13 전용 환경에서 저장소의 잠긴 의존성을 설치합니다. Colab 커널과 학습 Python은 별개입니다.
- `uv sync --frozen`으로 고정된 원격 anime-tools 그룹을 설치합니다. T4에서는 설치 직후 FlashAttention을 제거해 텍스트 캐시와 LLM adapter가 SDPA를 사용하도록 합니다.
- 기본 학습 HP는 anima-lora CLI에서 가져옵니다. T4 시작값은 512, batch 1, swap 8, checkpointing, SDPA, compile 끄기입니다. 폼에서 변경할 수 있습니다.
- BF16을 유지합니다. T4의 실제 지원 여부는 설치된 CUDA/PyTorch와 연산에 따라 달라지므로 GPU 검사와 실제 학습 smoke test를 모두 실행합니다.
- smoke test는 선택한 데이터·batch·해상도·학습 기법으로 짧게 학습합니다. 모든 버킷의 메모리 적합성이나 장기 학습 품질을 보장하지 않습니다.
- `/content`는 런타임 종료 시 사라집니다. 모델과 상태를 보존하려면 출력 폴더를 마운트한 Drive로 설정합니다.

기준 소스: [anima-lora](https://github.com/sorryhyun/anima_lora), [기존 diffusion-pipe 노트북](https://github.com/InoriNatsume/anima_colab_test/blob/main/anima_lora_train_colab.ipynb).

In [ ]:
#@title 1. 작업 경로와 설치
REPO_REVISION = "2b5d618619e3c58f503425b6fa6e47487f35a687" #@param {type:"string"}
WORK_ROOT = "/content/anima_cli" #@param {type:"string"}
MOUNT_DRIVE = False #@param {type:"boolean"}

import os, sys, json, subprocess, hashlib, shutil, tarfile, zipfile, stat, time, re, math
import urllib.request
import urllib.parse
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from copy import deepcopy

if not Path('/content').is_dir():
    raise RuntimeError('Google Colab의 GPU 런타임에서 실행하세요.')
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT = Path(WORK_ROOT).expanduser().resolve()
ROOT.mkdir(parents=True, exist_ok=True)
TOOLS = ROOT / 'tools'
TOOLS.mkdir(exist_ok=True)
ENV = os.environ.copy()
ENV['PYTHONUNBUFFERED'] = '1'
ENV['HF_HUB_DISABLE_TELEMETRY'] = '1'
PROCESSES = globals().get('PROCESSES', {})

def run(cmd, *, cwd=None, capture=False, log=None):
    cmd = [str(x) for x in cmd]
    if capture:
        result = subprocess.run(cmd, cwd=cwd, env=ENV, text=True, capture_output=True)
        if result.returncode:
            print(result.stdout[-6000:]); print(result.stderr[-6000:])
            raise RuntimeError(f'명령 실패({result.returncode}): {cmd[0]}')
        return result.stdout
    proc = subprocess.Popen(cmd, cwd=cwd, env=ENV, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True)
    handle = Path(log).open('w', encoding='utf-8') if log else None
    try:
        for line in proc.stdout:
            print(line, end='')
            if handle:
                handle.write(line); handle.flush()
        if proc.wait():
            raise RuntimeError(f'명령 실패({proc.returncode}). 위 로그를 확인하세요.')
    except BaseException:
        import signal
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGTERM)
            try: proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL); proc.wait()
        raise
    finally:
        if handle: handle.close()

def fetch_json(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'anima-colab'})
    with urllib.request.urlopen(req, timeout=60) as response:
        return json.load(response)

def safe_unzip(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as z:
        for info in z.infolist():
            name = info.filename.replace('\\', '/')
            target = (destination / name).resolve()
            if (PurePosixPath(name).is_absolute() or not target.is_relative_to(destination)
                    or stat.S_ISLNK(info.external_attr >> 16)):
                raise ValueError(f'허용되지 않는 ZIP 항목: {name}')
        z.extractall(destination)

if not re.fullmatch(r'[0-9a-fA-F]{40}', REPO_REVISION):
    raise ValueError('재현 가능한 설치를 위해 REPO_REVISION에 40자리 Git commit SHA를 넣으세요.')
REPO = ROOT / ('anima_lora_' + REPO_REVISION[:12])
if not REPO.exists():
    archive = ROOT / f'{REPO_REVISION}.zip'
    urllib.request.urlretrieve(f'https://codeload.github.com/sorryhyun/anima_lora/zip/{REPO_REVISION}', archive)
    safe_unzip(archive, ROOT)
    (ROOT / f'anima_lora-{REPO_REVISION}').rename(REPO)
ENV['ANIMA_HOME'] = str(REPO)
ENV['ANIME_TOOLS_HOME'] = str(REPO)
UV = TOOLS / 'uv'
if not UV.exists():
    release = fetch_json('https://api.github.com/repos/astral-sh/uv/releases/latest')
    asset = next(x for x in release['assets'] if x['name'] == 'uv-x86_64-unknown-linux-gnu.tar.gz')
    archive = TOOLS / 'uv.tar.gz'
    urllib.request.urlretrieve(asset['browser_download_url'], archive)
    with tarfile.open(archive) as tar:
        member = next(x for x in tar.getmembers() if x.isfile() and PurePosixPath(x.name).name == 'uv')
        with tar.extractfile(member) as src, UV.open('wb') as dst: shutil.copyfileobj(src, dst)
    UV.chmod(0o755)
# lock에 고정된 원격 anime-tools만 설치합니다.
# --locked는 제외된 개발 그룹의 ../anime_tools 메타데이터까지 확인합니다.
run([UV, 'sync', '--frozen', '--python', '3.13',
     '--no-default-groups', '--group', 'anime-tools-git'], cwd=REPO)
PY = REPO / '.venv/bin/python'
def py(source, *args, capture=False):
    return run([PY, '-c', source, *args], cwd=REPO, capture=capture)
def write_toml(path, data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    py('import json,sys,toml; from pathlib import Path; Path(sys.argv[1]).write_text(toml.dumps(json.loads(sys.argv[2])), encoding="utf-8")',
       str(path), json.dumps(data, ensure_ascii=False))
def read_toml(path):
    return json.loads(py('import json,sys,tomllib; print(json.dumps(tomllib.load(open(sys.argv[1],"rb"))))', str(path), capture=True))
run(['nvidia-smi'])

# 설치를 다시 실행해도 T4 호환성 처리를 자동 적용합니다.
gpu = json.loads(py("import json,torch,importlib.util; print(json.dumps({'major': torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else None, 'flash_installed': importlib.util.find_spec('flash_attn') is not None}))", capture=True))
if gpu['major'] is not None and gpu['major'] < 8:
    if gpu['flash_installed']:
        run([UV, 'pip', 'uninstall', '--python', PY, 'flash-attn'], cwd=REPO)
    py("from networks import attention_dispatch as a; assert a.flash_attn_varlen_func is None; assert a.flash_attn_func is None; print('T4: 텍스트 캐시용 SDPA 전환 확인 완료')")
else:
    print('FlashAttention 자동 제거 대상 아님. GPU capability major:', gpu['major'])

SMOKE_OK = None
(ROOT / 'environment.txt').write_text(run([UV, 'pip', 'freeze', '--python', PY], capture=True), encoding='utf-8')
print('설치 완료:', REPO, '\n학습 Python:', PY)

## 2. GPU 실행 확인

CUDA 초기화, BF16/FP16 행렬 연산, SDPA 역전파를 설치된 학습 환경에서 확인합니다.
설치 또는 CUDA 검사 실패 시 로그의 드라이버·PyTorch 버전부터 확인합니다. HP를 바꾸어도 설치 호환성 문제는 해결되지 않습니다.

In [ ]:
#@title GPU·연산 진단
py(r"""
import torch, json
if not torch.cuda.is_available():
    raise RuntimeError('CUDA를 사용할 수 없습니다. GPU 런타임과 드라이버/휠 호환성을 확인하세요.')
print('torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(), 'capability:', torch.cuda.get_device_capability())
print('BF16 native:', torch.cuda.is_bf16_supported(including_emulation=False))
for dtype in (torch.bfloat16, torch.float16):
    try:
        q = torch.randn(1, 4, 128, 64, device='cuda', dtype=dtype, requires_grad=True)
        result = torch.nn.functional.scaled_dot_product_attention(q, q, q)
        result.float().square().mean().backward()
        assert torch.isfinite(result).all() and torch.isfinite(q.grad).all()
        torch.cuda.synchronize()
        print(dtype, 'SDPA forward/backward PASS')
    except Exception as exc:
        print(dtype, 'FAIL:', type(exc).__name__, str(exc))
""")

In [ ]:
#@title 3. 데이터 ZIP 업로드 또는 기존 폴더
DATA_MODE = "upload_zip" #@param ["upload_zip", "zip_path", "folder"]
DATA_PATH = "" #@param {type:"string"}

if DATA_MODE == 'upload_zip':
    from google.colab import files
    uploads = files.upload()
    if len(uploads) != 1: raise ValueError('이미지·캡션 ZIP 하나를 선택하세요.')
    name, content = next(iter(uploads.items()))
    archive = ROOT / Path(name).name
    archive.write_bytes(content)
    del uploads, content
elif DATA_MODE == 'zip_path':
    archive = Path(DATA_PATH).expanduser().resolve()
elif DATA_MODE != 'folder':
    raise ValueError(DATA_MODE)
if DATA_MODE == 'folder':
    SOURCE = Path(DATA_PATH).expanduser().resolve()
else:
    if not archive.is_file() or not zipfile.is_zipfile(archive): raise ValueError('ZIP 경로를 확인하세요.')
    digest = hashlib.sha256()
    with archive.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''): digest.update(chunk)
    SOURCE = ROOT / 'datasets' / digest.hexdigest()[:16]
    if not (SOURCE / '.extracted').exists():
        SOURCE.mkdir(parents=True, exist_ok=True)
        safe_unzip(archive, SOURCE)
        (SOURCE / '.extracted').touch()
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
def image_files(folder):
    return sorted(p for p in Path(folder).rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS
                  and '__MACOSX' not in p.parts and not p.name.startswith('._'))
images = image_files(SOURCE)
if not images: raise ValueError('학습 이미지가 없습니다.')
missing = [str(p.relative_to(SOURCE)) for p in images if not p.with_suffix('.txt').is_file()]
if missing: raise ValueError(f'동일 이름 .txt 캡션 누락 {len(missing)}개: {missing[:10]}')
stems = [p.stem.casefold() for p in images]
if len(stems) != len(set(stems)): raise ValueError('캐시 이름 충돌을 피하도록 이미지 stem을 고유하게 바꾸세요.')
print('원본:', SOURCE, '| 이미지·캡션 쌍:', len(images))

## 4. TOML 생성 / 불러오기

`form`은 CLI 기본 설정을 바탕으로 아래 값을 적용합니다. `load`는 기존 anima-lora TOML과 `base_config` 상속을 읽으며 폼 HP를 적용하지 않습니다.
기존 파일의 상대 경로는 학습 저장소 기준입니다. 데이터는 한 subset을 사용하며, 기존 파일의 `image_dir`·`cache_dir`에 전처리합니다.

| 기법 | 켜기 | 끄기 |
|---|---|---|
| SVD 초기화 | `down_init = "weight_svd"` | `down_init = "kaiming"` |
| T-LoRA | `use_timestep_mask = true` | `false` |
| REPA | `use_repa = true` | `false` |

SVD는 초기화 방식이며, T-LoRA와 REPA는 각각 독립 옵션입니다. REPA를 켜면 `pe_spatial` 특징을 추가로 캐시합니다.
`low_vram`의 Unsloth offload와 block swap은 동시에 사용할 수 없습니다.

In [ ]:
#@title TOML 폼 — load 모드에서는 아래 HP를 덮어쓰지 않음
CONFIG_MODE = "form" #@param ["form", "load"]
EXISTING_TOML = "" #@param {type:"string"}
RUN_NAME = "anima_lora" #@param {type:"string"}
OUTPUT_ROOT = "/content/anima_cli/output" #@param {type:"string"}
RESOLUTION = 512 #@param [512, 768, 896, 1024] {type:"raw"}
BATCH_SIZE = 1 #@param {type:"integer"}
NUM_REPEATS = 2 #@param {type:"integer"}
GRAD_ACCUM = 1 #@param {type:"integer"}
RANK = 32 #@param {type:"integer"}
ALPHA = 128 #@param {type:"integer"}
OPTIMIZER = "AdamW" #@param ["AdamW", "Prodigy"]
LEARNING_RATE = 0.00002 #@param {type:"number"}
LR_SCHEDULER = "cosine" #@param ["cosine", "constant", "constant_with_warmup"]
WARMUP = 0.05 #@param {type:"number"}
TRAIN_LIMIT = "steps" #@param ["steps", "epochs"]
MAX_STEPS = 1000 #@param {type:"integer"}
MAX_EPOCHS = 8 #@param {type:"integer"}
SAVE_EVERY_STEPS = 250 #@param {type:"integer"}
SAVE_EVERY_EPOCHS = 1 #@param {type:"integer"}
SAVE_STATE = True #@param {type:"boolean"}
SEED = 42 #@param {type:"integer"}
MIXED_PRECISION = "bf16" #@param ["bf16", "fp16", "no"]
BLOCKS_TO_SWAP = 8 #@param {type:"integer"}
GRADIENT_CHECKPOINTING = True #@param {type:"boolean"}
UNSLOTH_OFFLOAD = False #@param {type:"boolean"}
TORCH_COMPILE = False #@param {type:"boolean"}
ATTENTION = "torch" #@param ["torch", "flash"]
SVD_INIT = True #@param {type:"boolean"}
T_LORA = True #@param {type:"boolean"}
MIN_RANK = 1 #@param {type:"integer"}
ALPHA_RANK_SCALE = 1.0 #@param {type:"number"}
REPA = True #@param {type:"boolean"}
REPA_WEIGHT = 0.05 #@param {type:"number"}
REPA_LAYER = 8 #@param {type:"integer"}
CAPTION_DROPOUT = 0.1 #@param {type:"number"}
SHUFFLE_VARIANTS = 4 #@param {type:"integer"}
CJK_VOCAB_PACK = False #@param {type:"boolean"}

if CONFIG_MODE == 'form':
    if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_NAME): raise ValueError('RUN_NAME: 영문·숫자·밑줄·하이픈을 사용하세요.')
    cfg = json.loads(py('import json; from library.config.io import load_method_preset; print(json.dumps(load_method_preset("lora","default")))', capture=True))
    cfg.update(network_dim=RANK, network_alpha=ALPHA, optimizer_type=OPTIMIZER,
        learning_rate=LEARNING_RATE, lr_scheduler=LR_SCHEDULER, lr_warmup_steps=WARMUP,
        gradient_accumulation_steps=GRAD_ACCUM, seed=SEED, mixed_precision=MIXED_PRECISION,
        blocks_to_swap=BLOCKS_TO_SWAP, gradient_checkpointing=GRADIENT_CHECKPOINTING,
        unsloth_offload_checkpointing=UNSLOTH_OFFLOAD, torch_compile=TORCH_COMPILE,
        attn_mode=ATTENTION, down_init='weight_svd' if SVD_INIT else 'kaiming',
        use_timestep_mask=T_LORA, min_rank=MIN_RANK, alpha_rank_scale=ALPHA_RANK_SCALE,
        use_repa=REPA, repa_weight=REPA_WEIGHT, repa_layer=REPA_LAYER,
        caption_dropout_rate=CAPTION_DROPOUT, use_shuffled_caption_variants=SHUFFLE_VARIANTS > 1,
        caption_shuffle_variants=SHUFFLE_VARIANTS, target_res=[RESOLUTION],
        save_state=SAVE_STATE, save_state_on_train_end=SAVE_STATE,
        save_every_n_steps=SAVE_EVERY_STEPS, save_every_n_epochs=SAVE_EVERY_EPOCHS,
        output_name=RUN_NAME, output_dir=str(Path(OUTPUT_ROOT) / RUN_NAME / 'ckpt'),
        logging_dir=str(Path(OUTPUT_ROOT) / RUN_NAME / 'logs'), log_with='tensorboard',
        log_every_n_steps=1, save_model_as='safetensors')
    # Epoch 제한은 max_train_steps를 덮어쓰므로 하나만 저장합니다.
    cfg.pop('max_train_epochs', None); cfg.pop('max_train_steps', None)
    cfg['max_train_steps' if TRAIN_LIMIT == 'steps' else 'max_train_epochs'] = MAX_STEPS if TRAIN_LIMIT == 'steps' else MAX_EPOCHS
    cfg.pop('checkpointing_epochs', None)
    if not CJK_VOCAB_PACK: cfg['vocab_pack'] = ''
    data_root = ROOT / 'prepared' / RUN_NAME
    cfg['resized_image_dir'] = str(data_root / 'resized')
    cfg['lora_cache_dir'] = str(data_root / 'cache')
    cfg['general'] = {}
    cfg['datasets'] = [dict(batch_size=BATCH_SIZE, validation_split_num=0, repeat_by_folder_name=False,
        subsets=[dict(image_dir=cfg['resized_image_dir'], cache_dir=cfg['lora_cache_dir'],
                      num_repeats=NUM_REPEATS, recursive=True)])]
else:
    if not Path(EXISTING_TOML).is_file(): raise ValueError('EXISTING_TOML 경로를 확인하세요.')
    cfg = json.loads(py(r"""
import json, sys, toml
from pathlib import Path
from library.config.io import _load_toml_with_base, load_dataset_config_from_base
p = str(Path(sys.argv[1]).resolve())
def inherited_blueprint(path, seen=None):
    seen = set() if seen is None else seen
    path = Path(path).resolve()
    if path in seen: raise ValueError('base_config cycle')
    seen.add(path)
    raw = toml.load(path)
    base = raw.get('base_config')
    result = inherited_blueprint(path.parent / base, seen) if base else {}
    result.update({k: raw[k] for k in ('general', 'datasets') if k in raw})
    return result
blueprint = inherited_blueprint(p)
c = _load_toml_with_base(p)
if c.get('dataset_config'):
    c.update(toml.load(c['dataset_config']))
    c.pop('dataset_config')
else:
    c.update(blueprint if blueprint.get('datasets') else (load_dataset_config_from_base(config_file=p, overrides=c) or {}))
print(json.dumps(c))
""", EXISTING_TOML, capture=True))

# 실행 제어 키는 standalone TOML에 포함하지 않습니다.
for key in ('method', 'preset', 'methods_subdir', 'config_file', 'print_config', 'output_config', 'config_snapshot'):
    cfg.pop(key, None)
if cfg.get('unsloth_offload_checkpointing') and cfg.get('blocks_to_swap', 0):
    raise ValueError('Unsloth offload와 block swap 중 하나만 선택하세요.')
if cfg.get('use_repa') and cfg.get('repa_encoder', 'pe_spatial') != 'pe_spatial':
    raise ValueError('이 전처리 경로의 REPA encoder는 pe_spatial입니다.')
datasets = cfg.get('datasets', [])
if len(datasets) != 1 or len(datasets[0].get('subsets', [])) != 1:
    raise ValueError('데이터 원본 하나에 대응하는 dataset/subset 하나를 지정하세요.')
subset = datasets[0]['subsets'][0]
def repo_path(value):
    p = Path(str(value)).expanduser()
    return str((p if p.is_absolute() else REPO / p).resolve())
for key in ('pretrained_model_name_or_path', 'qwen3', 'vae', 'output_dir', 'logging_dir'):
    if not cfg.get(key): raise ValueError(f'필수 설정 누락: {key}')
    cfg[key] = repo_path(cfg[key])
if cfg.get('vocab_pack'): cfg['vocab_pack'] = repo_path(cfg['vocab_pack'])
for key in ('image_dir', 'cache_dir'):
    if not subset.get(key): raise ValueError(f'dataset subset에 {key}가 필요합니다.')
    subset[key] = repo_path(str(subset[key]).format_map(cfg))
if any(Path(subset[k]) == SOURCE or Path(subset[k]).is_relative_to(SOURCE) or SOURCE.is_relative_to(Path(subset[k]))
       for k in ('image_dir', 'cache_dir')):
    raise ValueError('전처리/캐시 폴더는 원본 폴더와 분리하세요.')
if cfg.get('optimizer_type', '').lower() == 'adamw' and cfg.get('learning_rate', 0) >= 0.01:
    raise ValueError('AdamW 학습률이 큽니다. Prodigy의 LR=1을 그대로 사용했는지 확인하세요.')
if cfg.get('max_train_epochs') and cfg.get('max_train_steps'):
    raise ValueError('max_train_epochs와 max_train_steps 중 하나만 지정하세요.')
if datasets[0].get('batch_size', 1) < 1 or cfg.get('gradient_accumulation_steps', 1) < 1:
    raise ValueError('batch와 accumulation은 1 이상이어야 합니다.')
cfg.setdefault('log_with', 'tensorboard')
cfg.setdefault('target_res', [RESOLUTION])
cfg.setdefault('caption_shuffle_variants', SHUFFLE_VARIANTS)
if cfg.get('resume'): cfg['resume'] = repo_path(cfg['resume'])
if cfg.get('log_with') != 'tensorboard':
    raise ValueError('TensorBoard 모니터링을 사용하려면 기존 TOML의 log_with를 tensorboard로 지정하세요.')
if not re.fullmatch(r'[A-Za-z0-9_-]+', str(cfg.get('output_name', 'anima'))):
    raise ValueError('output_name: 영문·숫자·밑줄·하이픈을 사용하세요.')
CONFIG = ROOT / 'configs' / f'{cfg.get("output_name", "anima")}.toml'
write_toml(CONFIG, cfg)
SMOKE_OK = None
print(CONFIG.read_text(encoding='utf-8'))
print('저장:', CONFIG)
run([PY, 'train.py', '--config_file', CONFIG, '--print-config'], cwd=REPO)

In [ ]:
#@title 5. 모델 다운로드 — HF 토큰은 Colab Secret의 HF_TOKEN에서 읽음
DOWNLOAD_MODELS = True #@param {type:"boolean"}
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None
if token: ENV['HF_TOKEN'] = token
del token
if DOWNLOAD_MODELS:
    run([PY, 'tasks.py', 'download-model', 'anima'], cwd=REPO)
    if cfg.get('vocab_pack'): run([PY, 'tasks.py', 'download-model', 'cjk'], cwd=REPO)
for key in ('pretrained_model_name_or_path', 'qwen3', 'vae'):
    if not Path(cfg[key]).is_file(): raise FileNotFoundError(f'{key}: {cfg[key]}')
print('모델 경로 확인 완료. REPA 특징 모델은 특징 캐시 단계에서 로드됩니다.')

## 6. 전처리

원본 비율을 보존해 지정 해상도 tier로 리사이즈하고 VAE·텍스트 캐시를 만듭니다. REPA 활성화 시 PE-Spatial 캐시를 추가합니다.
원본·캡션·모델·해상도·셔플 설정이 바뀌면 기존 캐시를 섞지 않도록 중단합니다. 새로운 실행 이름/캐시 경로를 지정하세요.

In [ ]:
#@title 리사이즈·VAE·텍스트·REPA 캐시
CACHE_BATCH_SIZE = 1 #@param {type:"integer"}
VAE_FP32 = True #@param {type:"boolean"}
VAE_CHUNK_SIZE = 64 #@param {type:"integer"}

def fingerprint():
    # 원본 데이터와 캐시 목록의 변경도 smoke 통과 기록에 반영합니다.
    h = hashlib.sha256(CONFIG.read_bytes())
    h.update(REPO_REVISION.encode())
    for key in ('pretrained_model_name_or_path', 'qwen3', 'vae'):
        p = Path(cfg[key]); s = p.stat()
        h.update(f'{p}:{s.st_size}:{s.st_mtime_ns}'.encode())
    for folder in (SOURCE, Path(subset['image_dir']), Path(subset['cache_dir'])):
        if folder.exists():
            for p in sorted(folder.rglob('*')):
                if p.is_file():
                    s = p.stat(); h.update(f'{p}:{s.st_size}:{s.st_mtime_ns}'.encode())
    return h.hexdigest()

def preparation_signature():
    source_hash = hashlib.sha256()
    for p in images:
        for file in (p, p.with_suffix('.txt')):
            source_hash.update(str(file.relative_to(SOURCE)).encode())
            with file.open('rb') as f:
                for block in iter(lambda: f.read(8 * 1024 * 1024), b''): source_hash.update(block)
    keys = ('pretrained_model_name_or_path', 'qwen3', 'vae', 'vocab_pack', 'target_res', 'caption_shuffle_variants',
            'caption_tag_dropout_rate', 'caption_tag_randomize_rate')
    return dict(source=source_hash.hexdigest(), repo=REPO_REVISION, config={k: cfg.get(k) for k in keys},
                vae_fp32=VAE_FP32, vae_chunk=VAE_CHUNK_SIZE,
                model_stats={k: [Path(cfg[k]).stat().st_size, Path(cfg[k]).stat().st_mtime_ns]
                             for k in keys[:3]})

image_dir, cache_dir = Path(subset['image_dir']), Path(subset['cache_dir'])
manifest = cache_dir / 'colab_preparation.json'
signature = preparation_signature()
if manifest.exists():
    if json.loads(manifest.read_text()) != signature:
        raise RuntimeError('전처리 조건이 변경됐습니다. 새로운 image_dir/cache_dir로 실행하세요.')
elif any(folder.exists() and any(folder.iterdir()) for folder in (image_dir, cache_dir)):
    raise RuntimeError('출처를 확인할 수 없는 전처리 파일이 있습니다. 비어 있는 별도 경로를 지정하세요.')
cache_dir.mkdir(parents=True, exist_ok=True)
# 중단 후 재시도할 때 동일 조건임을 확인할 수 있도록 캐시 생성 전에 기록합니다.
manifest.write_text(json.dumps(signature, indent=2, ensure_ascii=False), encoding='utf-8')
prep = dict(source_image_dir=str(SOURCE), resized_image_dir=str(image_dir), lora_cache_dir=str(cache_dir),
            target_res=cfg['target_res'])
prep_path = ROOT / 'configs' / 'preprocess_paths.toml'
write_toml(prep_path, prep)
ENV['CONFIG_FILE'] = str(prep_path)
run([PY, 'tasks.py', 'preprocess-resize', '--target_res', *map(str, cfg['target_res'])], cwd=REPO)
ENV.pop('CONFIG_FILE', None)
common = ['--cache_dir', cache_dir, '--batch_size', CACHE_BATCH_SIZE, '--recursive']
run([PY, 'scripts/preprocess/cache_latents.py', '--dir', image_dir, '--vae', cfg['vae'],
     '--chunk_size', VAE_CHUNK_SIZE, *common, *(['--no_half_vae'] if VAE_FP32 else [])], cwd=REPO)
run([PY, 'scripts/preprocess/cache_text_embeddings.py', '--dir', SOURCE,
     '--match_images_from', image_dir, '--qwen3', cfg['qwen3'], '--dit', cfg['pretrained_model_name_or_path'],
     '--vocab_pack', cfg.get('vocab_pack', ''), '--caption_shuffle_variants', cfg['caption_shuffle_variants'],
     '--caption_tag_dropout_rate', cfg.get('caption_tag_dropout_rate', 0.0),
     '--caption_tag_randomize_rate', cfg.get('caption_tag_randomize_rate', 0.0),
     *common], cwd=REPO)
if cfg.get('use_repa'):
    run([PY, 'scripts/preprocess/cache_pe_encoder.py', '--dir', image_dir,
         '--encoder', 'pe_spatial', '--dtype', 'float32', *common], cwd=REPO)
prepared = image_files(image_dir)
if len(prepared) != len(images):
    raise RuntimeError(f'원본 {len(images)}장 / 리사이즈 {len(prepared)}장. 전처리 로그를 확인하세요.')
PREPARED_SIGNATURE = signature
SMOKE_OK = None
print('전처리 완료:', len(prepared), '장')

## 7. Smoke test

모델 로딩 → LoRA 생성·SVD 초기화 → 5-step 학습 → 체크포인트 저장·검사 순서로 실행합니다.
`Creating DiT LoRA: n/280`은 모듈 생성 진행률입니다. 초기화 시간은 5-step 학습 시간에 추가됩니다.

본 학습 TOML에서 학습량·출력·로그·저장·재개 관련 설정만 smoke용으로 바꿉니다. batch, accumulation, 해상도, SVD, T-LoRA, REPA는 유지합니다.
정상 종료, 완료 step, 유한한 loss, 저장된 LoRA tensor의 유한성을 검사합니다. loss의 증가·감소는 통과 조건에 포함하지 않습니다.


In [ ]:
#@title 현재 설정으로 짧은 실제 학습
SMOKE_STEPS = 5 #@param {type:"integer"}
if SMOKE_STEPS < 2: raise ValueError('SMOKE_STEPS는 2 이상으로 설정하세요.')
if read_toml(CONFIG) != cfg: raise RuntimeError('TOML이 변경됐습니다. 설정 셀부터 다시 실행하세요.')
if preparation_signature() != PREPARED_SIGNATURE: raise RuntimeError('데이터/모델 변경: 전처리 셀부터 다시 실행하세요.')
SMOKE_OK = None
smoke = deepcopy(cfg)
for key in ('max_train_epochs', 'resume', 'initial_epoch', 'initial_step', 'checkpointing_epochs', 'save_every_n_epochs'):
    smoke.pop(key, None)
smoke_dir = ROOT / 'smoke' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
smoke_dir.mkdir(parents=True)
progress = smoke_dir / 'progress.jsonl'
smoke.update(max_train_steps=SMOKE_STEPS, output_dir=str(smoke_dir), output_name='smoke',
             logging_dir=str(smoke_dir / 'logs'), log_with='tensorboard', log_every_n_steps=1,
             progress_jsonl=str(progress), save_every_n_steps=SMOKE_STEPS,
             save_state=False, save_state_on_train_end=False, save_model_as='safetensors')
smoke_path = smoke_dir / 'smoke.toml'
write_toml(smoke_path, smoke)
before = fingerprint()
print(f'모델·LoRA 초기화 후 {SMOKE_STEPS} optimizer steps를 학습하고 저장합니다.')
print('loss 검사는 NaN/Inf 여부를 확인하며, 증가 추세는 실패 조건이 아닙니다.')
run([PY, 'train.py', '--config_file', smoke_path], cwd=REPO, log=smoke_dir / 'console.log')
records = [json.loads(line) for line in progress.read_text().splitlines() if line.strip()]
steps = [r for r in records if r.get('ev') == 'step']
ends = [r for r in records if r.get('ev') == 'run_end']
losses = [v for r in steps for k, v in r.items() if 'loss' in k.lower()]
if not ends or ends[-1].get('status') != 'ok' or ends[-1].get('final_step', 0) < SMOKE_STEPS:
    raise RuntimeError('smoke 완료 step/종료 상태 검증 실패')
if not losses or not all(isinstance(v, (int, float)) and math.isfinite(v) for v in losses):
    raise RuntimeError('loss가 없거나 NaN/Inf입니다.')
py(r"""
import sys, torch
from pathlib import Path
from safetensors import safe_open
files = list(Path(sys.argv[1]).glob('smoke*.safetensors'))
if not files: raise RuntimeError('저장된 LoRA가 없습니다.')
count = 0
for path in files:
    with safe_open(path, framework='pt', device='cpu') as f:
        for key in f.keys():
            if not torch.isfinite(f.get_tensor(key)).all(): raise RuntimeError(f'NaN/Inf: {path.name}:{key}')
            count += 1
if not count: raise RuntimeError('빈 체크포인트입니다.')
print('저장 tensor 검사 PASS:', count)
""", str(smoke_dir))
if fingerprint() != before: raise RuntimeError('smoke 중 설정/데이터가 변경됐습니다.')
SMOKE_OK = before
report = dict(status='PASS', steps=SMOKE_STEPS, fingerprint=before, repo=REPO_REVISION,
              loss_min=min(losses), loss_max=max(losses), time=datetime.now(timezone.utc).isoformat())
(smoke_dir / 'result.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print('SMOKE PASS:', smoke_dir)

## 8. TensorBoard / cloudflared

기본값은 Colab 내부 보기입니다. `PUBLIC_TUNNEL`을 켜면 인증 없는 임시 공개 URL이 생성되며, URL을 아는 사람은 학습 로그를 볼 수 있습니다.
cloudflared는 TensorBoard만 연결합니다. 사용 후 마지막 셀에서 종료할 수 있습니다.

In [ ]:
#@title TensorBoard 시작 및 선택적 공개 터널
TB_PORT = 6006 #@param {type:"integer"}
PUBLIC_TUNNEL = False #@param {type:"boolean"}
def stop_service(name):
    service = PROCESSES.pop(name, None)
    if service:
        proc, handle = service
        if proc.poll() is None:
            proc.terminate()
            try: proc.wait(timeout=10)
            except subprocess.TimeoutExpired: proc.kill(); proc.wait()
        handle.close()
def start_service(name, command):
    stop_service(name)
    log = ROOT / f'{name}.log'
    handle = log.open('w')
    proc = subprocess.Popen([str(x) for x in command], cwd=REPO, env=ENV,
                            stdout=handle, stderr=subprocess.STDOUT)
    PROCESSES[name] = (proc, handle)
    return proc, log
stop_service('cloudflared')
Path(cfg['logging_dir']).mkdir(parents=True, exist_ok=True)
tb, tb_log = start_service('tensorboard', [PY, '-m', 'tensorboard.main', '--logdir', cfg['logging_dir'],
                                          '--host', '127.0.0.1', '--port', TB_PORT])
for _ in range(60):
    if tb.poll() is not None: raise RuntimeError(tb_log.read_text())
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{TB_PORT}/', timeout=2).close()
        break
    except OSError: time.sleep(1)
else: raise RuntimeError('TensorBoard 시작 시간 초과: ' + str(tb_log))
from google.colab import output
output.serve_kernel_port_as_iframe(TB_PORT, height=700)
if PUBLIC_TUNNEL:
    binary = TOOLS / 'cloudflared'
    if not binary.exists():
        release = fetch_json('https://api.github.com/repos/cloudflare/cloudflared/releases/latest')
        asset = next(a for a in release['assets'] if a['name'] == 'cloudflared-linux-amd64')
        urllib.request.urlretrieve(asset['browser_download_url'], binary)
        binary.chmod(0o755)
    tunnel, tunnel_log = start_service('cloudflared', [binary, 'tunnel', '--url', f'http://127.0.0.1:{TB_PORT}', '--no-autoupdate'])
    for _ in range(60):
        if tunnel.poll() is not None: raise RuntimeError(tunnel_log.read_text())
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', tunnel_log.read_text())
        if match:
            print('공개 TensorBoard:', match.group()); break
        time.sleep(1)
    else: raise RuntimeError('터널 주소 발급 시간 초과: ' + str(tunnel_log))

In [ ]:
#@title 9. 본 학습 — smoke PASS 후 실행
START_TRAINING = False #@param {type:"boolean"}
if START_TRAINING:
    if not SMOKE_OK or SMOKE_OK != fingerprint():
        raise RuntimeError('현재 설정/데이터로 smoke test를 통과해야 합니다.')
    if read_toml(CONFIG) != cfg:
        raise RuntimeError('TOML이 변경됐습니다. 설정 셀부터 다시 실행하세요.')
    output_dir = Path(cfg['output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)
    if any(output_dir.glob('*.safetensors')) and not cfg.get('resume'):
        raise RuntimeError('기존 체크포인트가 있습니다. 새 출력 경로 또는 명시적인 resume을 사용하세요.')
    saved = output_dir / f'{cfg.get("output_name", "anima")}_colab.toml'
    shutil.copy2(CONFIG, saved)
    log = output_dir / ('train_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.log')
    run([PY, 'train.py', '--config_file', CONFIG], cwd=REPO, log=log)
    print('학습 완료:', output_dir)
else:
    print('START_TRAINING을 켜고 이 셀을 실행하면 본 학습을 시작합니다.')

In [ ]:
#@title 10. TOML 내려받기 / 모니터링 종료
DOWNLOAD_TOML = False #@param {type:"boolean"}
STOP_MONITORING = False #@param {type:"boolean"}
if DOWNLOAD_TOML:
    from google.colab import files
    files.download(str(CONFIG))
if STOP_MONITORING:
    stop_service('cloudflared'); stop_service('tensorboard')
    print('모니터링 종료')
print('TOML:', CONFIG)
print('체크포인트:', cfg['output_dir'])

## 설정 조정

- **CUDA/의존성 실패:** 설치 로그와 GPU 진단 결과를 확인합니다. 고정한 소스의 CUDA 휠이 현재 Colab 드라이버와 맞아야 합니다.
- **OOM:** batch, 해상도, swap, REPA 중 조정할 항목을 폼에서 변경한 후 전처리·smoke를 다시 실행합니다. Unsloth offload와 swap은 상호 배타적입니다.
- **NaN/Inf:** smoke 로그와 정밀도·학습률을 확인합니다. Prodigy의 LR=1과 AdamW의 LR=2e-5는 서로 다른 설정입니다.
- **해상도 변경:** 학습 TOML만 바꾸어도 기존 latent 크기는 바뀌지 않습니다. 새로운 캐시 경로에서 전처리합니다.
- **이어 학습:** 저장된 optimizer state 디렉터리를 기존 TOML의 `resume`에 지정하고 `load`로 불러옵니다. smoke는 resume을 제거해 새로 검사하고 본 학습은 원래 resume을 사용합니다.
- **학습량:** 기본 1,000 steps는 조절 가능한 출발값입니다. 데이터 수·반복·batch에 따라 적정 학습량은 달라집니다.

[uv 환경 동기화](https://docs.astral.sh/uv/concepts/projects/sync/) · [Cloudflare Quick Tunnels](https://developers.cloudflare.com/cloudflare-one/networks/connectors/cloudflare-tunnel/do-more-with-tunnels/trycloudflare/)